# Exploratory Data Analysis: GitHub Repository Quality Predictor

## Deep-Dive Statistical Analysis, Power-Law Distributions & Feature Dynamics



### Project Context

This notebook performs an exhaustive Exploratory Data Analysis (EDA) on the raw scraped dataset of **19,179 GitHub repositories** collected across 8 popularity tiers, 4 recency windows, and 3 primary programming languages (Python, JavaScript, TypeScript).



### Core Objectives:

1. **Audit Data Integrity**: Identify missingness patterns, duplicates, and data types.

2. **Characterize Distributions**: Uncover extreme skew and power-law dynamics across community metrics.

3. **Detect Multicollinearity**: Statistically prove redundant features (e.g., `watchers_count` vs `stargazers_count`).

4. **Analyze Temporal Dynamics**: Measure repository age and abandonment cliffs (`days_since_last_push`).

5. **Synthesize Modeling Directives**: Derive mathematical transformations and rubric weights for feature engineering.

### 1. Environment Setup & Data Ingestion

Configure visual styles, plotting defaults, and load the raw dataset (`data/raw/github_repos.csv`).

In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



# Visual theme configuration

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

plt.rcParams['figure.figsize'] = (12, 6)

plt.rcParams['font.size'] = 10

plt.rcParams['axes.titlesize'] = 12

plt.rcParams['axes.titleweight'] = 'bold'

plt.rcParams['figure.dpi'] = 120



# Robust data path handling

data_path = Path('../data/raw/github_repos.csv')

if not data_path.exists():

    data_path = Path('data/raw/github_repos.csv')



df = pd.read_csv(data_path)

print(f"Raw dataset ingested successfully: {df.shape[0]:,} rows, {df.shape[1]} columns.")

df.head(3)

### 2. Dataset Dimensions & Schema Audit

Inspect column data types, non-null counts, and memory footprint.

In [ ]:
schema_audit = pd.DataFrame({

    'Data Type': df.dtypes,

    'Non-Null Count': df.notnull().sum(),

    'Null Count': df.isnull().sum(),

    'Null %': (df.isnull().sum() / len(df) * 100).round(2),

    'Unique Values': df.nunique()

})

print(f"Total Memory Usage: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

schema_audit

### 3. Missing Value Analysis

Evaluate columns with missing data and determine the domain rationale:

- `license_key`: Missing in ~33% of repositories because many repositories do not specify an open-source license.

- `language`: Missing in 12 repositories where GitHub could not detect a dominant programming language (e.g., pure documentation or asset repos).

In [ ]:
null_cols = schema_audit[schema_audit['Null Count'] > 0]



fig, ax = plt.subplots(figsize=(8, 4))

sns.barplot(x=null_cols.index, y=null_cols['Null %'], palette='Reds_r', ax=ax)

ax.set_title('Missing Value Percentage by Column')

ax.set_ylabel('Missing Percentage (%)')

ax.set_ylim(0, 40)



for p in ax.patches:

    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 1.0),

                ha='center', va='center', fontsize=10, color='black')

plt.tight_layout()

plt.show()

### 4. Univariate Distributions: The Power-Law Reality of Open Source

Community metrics like stars, forks, and issues follow extreme **heavy-tailed / power-law distributions**.

A tiny minority of legendary repositories receive tens of thousands of stars, while the vast majority sit below 100 stars.

This proves that **linear metrics fail** and justifies our logarithmic transformation: $\log_{10}(1 + x)$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))



# Raw Stars vs Log Stars

sns.histplot(df['stargazers_count'], bins=50, ax=axes[0, 0], color='#1f77b4', kde=False)

axes[0, 0].set_title('Raw Stars Distribution (Extreme Right Skew)')

axes[0, 0].set_xlabel('Stargazers Count')



sns.histplot(np.log10(1 + df['stargazers_count']), bins=40, ax=axes[0, 1], color='#1f77b4', kde=True)

axes[0, 1].set_title('Log-Transformed Stars: log10(1 + Stars)')

axes[0, 1].set_xlabel('log10(1 + Stargazers)')



# Raw Forks vs Log Forks

sns.histplot(df['forks_count'], bins=50, ax=axes[1, 0], color='#2ca02c', kde=False)

axes[1, 0].set_title('Raw Forks Distribution (Extreme Right Skew)')

axes[1, 0].set_xlabel('Forks Count')



sns.histplot(np.log10(1 + df['forks_count']), bins=40, ax=axes[1, 1], color='#2ca02c', kde=True)

axes[1, 1].set_title('Log-Transformed Forks: log10(1 + Forks)')

axes[1, 1].set_xlabel('log10(1 + Forks)')



plt.tight_layout()

plt.show()



# Descriptive Quantiles

print("--- Stargazers & Forks Quantiles ---")

display(df[['stargazers_count', 'forks_count', 'open_issues_count', 'size_kb']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(2))

### 5. Temporal Dynamics: Age & The Abandonment Cliff

We convert ISO timestamp strings into concrete time intervals:

- `repo_age_days`: Time elapsed since repository creation.

- `days_since_last_push`: Inactivity metric measuring the liveness and maintenance frequency.

In [ ]:
now = pd.Timestamp.now(tz="UTC")

df['repo_age_days'] = (now - pd.to_datetime(df['created_at'], utc=True)).dt.days

df['days_since_last_push'] = (now - pd.to_datetime(df['pushed_at'], utc=True)).dt.days



fig, axes = plt.subplots(1, 2, figsize=(14, 5))



sns.histplot(df['repo_age_days'] / 365.25, bins=35, ax=axes[0], color='#8c564b', kde=True)

axes[0].set_title('Repository Age Distribution (Years)')

axes[0].set_xlabel('Age in Years')



sns.histplot(df['days_since_last_push'], bins=40, ax=axes[1], color='#d62728', kde=False)

axes[1].set_title('Inactivity Distribution (Days Since Last Push)')

axes[1].set_xlabel('Days Inactive')



plt.tight_layout()

plt.show()



# Abandonment thresholds

active_30 = (df['days_since_last_push'] <= 30).mean() * 100

stale_180 = ((df['days_since_last_push'] > 30) & (df['days_since_last_push'] <= 180)).mean() * 100

abandoned_365 = (df['days_since_last_push'] > 365).mean() * 100



print(f"Maintenance Recency Breakdown:")

print(f"  - Active (pushed within 30 days):   {active_30:.1f}%")

print(f"  - Cooling (30 to 180 days idle):    {stale_180:.1f}%")

print(f"  - Abandoned (inactive > 1 year):    {abandoned_365:.1f}%")

### 6. Ecosystem & Categorical Distributions

Analyzing programming languages, organization ownership, and open-source licensing adoption.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))



# 1. Language Breakdown

lang_counts = df['language'].value_counts().head(5)

sns.barplot(x=lang_counts.index, y=lang_counts.values, ax=axes[0], palette='Blues_r')

axes[0].set_title('Target Programming Languages')

axes[0].set_ylabel('Number of Repositories')



# 2. Organization vs User Ownership

owner_counts = df['owner_type'].value_counts()

axes[1].pie(owner_counts, labels=owner_counts.index, autopct='%1.1f%%', colors=['#4a90e2', '#50e3c2'], startangle=90)

axes[1].set_title('Owner Governance (User vs Org)')



# 3. Top Licenses

lic_counts = df['license_key'].fillna('None / Unspecified').value_counts().head(6)

sns.barplot(x=lic_counts.values, y=lic_counts.index, ax=axes[2], palette='viridis')

axes[2].set_title('Top Open-Source Licenses')

axes[2].set_xlabel('Count')



plt.tight_layout()

plt.show()

### 7. Governance Impact: Does Organization Backing Drive Higher Traction?

We compare star and fork distributions between **Organization** and **Individual User** repositories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))



sns.boxplot(x='owner_type', y=np.log10(1 + df['stargazers_count']), data=df, ax=axes[0], palette='Set2')

axes[0].set_title('Log Stars by Governance Type')

axes[0].set_ylabel('log10(1 + Stargazers)')



sns.boxplot(x='owner_type', y=np.log10(1 + df['forks_count']), data=df, ax=axes[1], palette='Set2')

axes[1].set_title('Log Forks by Governance Type')

axes[1].set_ylabel('log10(1 + Forks)')



plt.tight_layout()

plt.show()



# Statistical comparison

org_median_stars = df[df['owner_type'] == 'Organization']['stargazers_count'].median()

user_median_stars = df[df['owner_type'] == 'User']['stargazers_count'].median()

print(f"Median Stars for Organizations: {org_median_stars:,.0f}")

print(f"Median Stars for Users:         {user_median_stars:,.0f} (Orgs have ~{org_median_stars / max(user_median_stars, 1):.1f}x higher median)")

### 8. Multicollinearity Discovery: `watchers_count` vs. `stargazers_count`

GitHub's API documentation notes that `watchers_count` is historically synonymous with `stargazers_count`.

Let's test their correlation empirically.

In [ ]:
corr_watchers = df['stargazers_count'].corr(df['watchers_count'])

exact_matches = (df['stargazers_count'] == df['watchers_count']).mean() * 100



print(f"Correlation between stargazers_count and watchers_count: {corr_watchers:.6f}")

print(f"Percentage of rows where stargazers == watchers:         {exact_matches:.2f}%")



fig, ax = plt.subplots(figsize=(7, 5))

sns.scatterplot(x='stargazers_count', y='watchers_count', data=df.sample(2000, random_state=42), ax=ax, alpha=0.5, color='#e74c3c')

ax.set_title('Multicollinearity Check: Stargazers vs Watchers (r = 1.000)')

ax.set_xlabel('stargazers_count')

ax.set_ylabel('watchers_count')

plt.tight_layout()

plt.show()



print(">> CRITICAL MODELING ACTION: Drop 'watchers_count' to eliminate perfect multicollinearity.")

### 9. Bivariate Relationship: Stars vs. Forks Dynamics

How do stars (bookmarks) relate to forks (code integrations)?

Healthy active software clusters along a consistent `fork_to_star_ratio` band.

In [ ]:
df['log_stars'] = np.log10(1 + df['stargazers_count'])

df['log_forks'] = np.log10(1 + df['forks_count'])

df['fork_to_star_ratio'] = (df['forks_count'] + 1) / (df['stargazers_count'] + 1)



fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(

    x='log_stars', y='log_forks', hue='owner_type',

    data=df.sample(3000, random_state=42), alpha=0.6, ax=ax, palette='tab10'

)

ax.set_title('Log-Log Relationship: Forks vs. Stars by Governance')

ax.set_xlabel('log10(1 + Stargazers)')

ax.set_ylabel('log10(1 + Forks)')

plt.tight_layout()

plt.show()

### 10. Comprehensive Correlation Matrix

Evaluating cross-feature correlation across all numerical and temporal signals.

In [ ]:
numeric_cols = [

    'stargazers_count', 'forks_count', 'open_issues_count', 'size_kb',

    'topics_count', 'description_length', 'repo_age_days', 'days_since_last_push',

    'log_stars', 'log_forks', 'fork_to_star_ratio'

]



corr_matrix = df[numeric_cols].corr()



fig, ax = plt.subplots(figsize=(11, 8))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax, linewidths=0.5)

ax.set_title('Numerical Feature Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()

plt.show()

### 11. Key EDA Discoveries & Modeling Roadmap



| Discovery | Statistical Evidence | Engineering Action in V2 Pipeline |

| :--- | :--- | :--- |

| **1. Extreme Right Skew** | Stars median is 50, but max is 495k | Apply $\log_{10}(1 + x)$ scaling to all volume counters |

| **2. Exact Collinearity** | $r = 1.000000$ between `watchers_count` and `stargazers_count` | Permanently drop `watchers_count` |

| **3. The Age Decay Bias** | Older repos accumulate passive stars | Engineer **log velocity** with $+30$d Bayesian prior to uncouple age |

| **4. Maintenance Cliff** | >40% of repos have no commits in >1 year | Introduce non-linear step penalties for `days_since_last_push` |

| **5. License Importance** | 33% of repos lack a license | Create explicit `license_type_osi` binary flag for compliance |

| **6. Org Governance Premium** | Orgs have 5x higher median stars | Add `owner_type_org` feature and governance rubric weight |